# 生命讀經開發測試資料＋找出要加入的教會專名（GPU）

用你 Google Drive 裡 `twgbr-archive` 的生命讀經**文字與音訊**：

1. 從各卷平均挑 16 篇（不用第 1 篇），複製到 `twgbr-archive/life-study-dev-v1/`
2. 用 Whisper large-v3-turbo 以「停頓切段 ≤10 秒」辨識（和網站的 Whisper 相同設定）
3. 從恢復本＋生命讀經的正確文字找出常用詞，檢查哪些詞加入「同音教會專名校正」後能修好辨識錯字、又不會改錯正確文字

**開始前**：「執行階段 → 變更執行階段類型 → T4 GPU」。由上而下逐格按 ▶，約 40–60 分鐘。

注意：生命讀經音訊是講者帶讀並加上講解，**不是網頁文字的逐字朗讀**，所以 CER 只供參考，以「專名比對（不需逐字對齊）」為準。
這是開發資料（不是 MEET 封存測試）；內容版權屬臺灣福音書房，只在你的雲端硬碟私人使用，不訓練模型。

In [ ]:
# 第 1 格：設定
ARCHIVE_FOLDER = 'twgbr-archive'   # 「我的雲端硬碟」裡的資料夾名稱
COUNT = 16                         # 挑幾篇生命讀經
CONDITIONS = 'vad10'               # 停頓切段、每段最長 10 秒（網站 Whisper 的做法）

In [ ]:
# 第 2 格：連接 Google Drive、找到資料、確認 GPU
import os, subprocess, shutil
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
ARCHIVE = ROOT/ARCHIVE_FOLDER
if not (ARCHIVE/'life-study').is_dir():
    found = [Path(d) for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'life-study' in dirs and 'recovery-version' in dirs]
    if not found:
        raise RuntimeError(f'找不到含 life-study 與 recovery-version 的 {ARCHIVE_FOLDER} 資料夾')
    ARCHIVE = found[0]
DEV = ARCHIVE/'life-study-dev-v1'
print('資料位置：', ARCHIVE)
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if shutil.which('nvidia-smi') else ''
print('GPU：', gpu or '沒有 GPU（請改選 T4 GPU 後從第 1 格重新執行）')

In [ ]:
# 第 3 格：取得固定版本的評估程式與套件（含 GPU 函式庫，約 2–3 分鐘）
import sys
COMMIT = '3adc25849886006e3330d6a15151e03c5db9c4c2'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faster-whisper==1.2.1', 'ctranslate2==4.8.3', 'av==16.0.1',
                'rapidfuzz==3.14.1', 'opencc-python-reimplemented==0.1.7', 'nodejs-wheel-binaries==22.20.0',
                # CTranslate2 needs CUDA 12 cuBLAS + cuDNN 9; newer Colab images do not ship libcublas.so.12.
                'nvidia-cublas-cu12==12.9.1.4', 'nvidia-cudnn-cu12==9.28.0.42'], check=True)
import importlib, importlib.util
importlib.invalidate_caches()
lib_dirs = [str(Path(location)/'lib') for module in ('nvidia.cublas', 'nvidia.cudnn')
            for location in importlib.util.find_spec(module).submodule_search_locations]
os.environ['LD_LIBRARY_PATH'] = ':'.join(lib_dirs + [os.environ.get('LD_LIBRARY_PATH', '')])
print('CUDA 12 函式庫：', *lib_dirs, sep='\n  ')
import nodejs_wheel
os.environ['NODE_BINARY'] = str(Path(nodejs_wheel.__file__).parent/'bin'/'node')
print('程式版本', COMMIT[:12], '｜Node', subprocess.check_output([os.environ['NODE_BINARY'], '--version'], text=True).strip())

In [ ]:
# 第 4 格：下載固定版本的 Whisper large-v3-turbo 並核對 SHA-256（約 1.6GB）
import hashlib
from huggingface_hub import snapshot_download
MODEL_DIR = snapshot_download('deepdml/faster-whisper-large-v3-turbo-ct2', revision='4df90f75321148c3a29a9e2351b7ddf8f5b115a8',
                              allow_patterns=['config.json', 'model.bin', 'preprocessor_config.json', 'tokenizer.json', 'vocabulary.json'])
digest = hashlib.sha256()
with open(Path(MODEL_DIR)/'model.bin', 'rb') as stream:
    for block in iter(lambda: stream.read(1 << 24), b''):
        digest.update(block)
if digest.hexdigest() != 'e76620f83d5f5b69efd3d87e3dc180c1bd21df9fbebacfd4335e5e1efcc018da':
    raise RuntimeError('模型檔 SHA-256 不符，已停止')
print('模型已核對：', MODEL_DIR)

In [ ]:
# 第 5 格：建立開發測試資料（複製 16 篇的文字與音訊，約 1–3 分鐘）
subprocess.run([sys.executable, str(REPO/'scripts/build_life_study_dev_set.py'), '--archive', str(ARCHIVE),
                '--output', str(DEV), '--count', str(COUNT)], check=True, cwd=REPO)

In [ ]:
# 第 6 格：Whisper 辨識與專名比對（GPU 約 30–40 分鐘；中斷後重新執行會沿用已完成的部分）
from datetime import datetime, timezone
RUNS = DEV/'eval-runs'
earlier = sorted(p for p in RUNS.glob('*') if p.is_dir()) if RUNS.is_dir() else []
RUN = RUNS/datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S')
command = [sys.executable, '-u', str(REPO/'scripts/evaluate_meet.py'), '--meet', str(DEV), '--output', str(RUN),
           '--model-dir', MODEL_DIR, '--model-label', 'whisper-large-v3-turbo (deepdml ct2 4df90f7)', '--conditions', CONDITIONS,
           '--development', '--loose-reference']
for folder in reversed(earlier):
    command += ['--reuse', str(folder)]
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('評估失敗，請看上面的錯誤訊息')

In [ ]:
# 第 7 格：找出值得加入的教會專名（約 5–10 分鐘）
# The miner converts Whisper's Simplified text with opencc-js (the same converter as the website's Worker):
# fetch the version pinned in package-lock.json and check its integrity hash.
import base64, hashlib, json, tarfile, urllib.request, io
lock = json.loads((REPO/'package-lock.json').read_text())['packages']['node_modules/opencc-js']
target = REPO/'node_modules/opencc-js'
if not (target/'package.json').exists():
    data = urllib.request.urlopen(lock['resolved'], timeout=120).read()
    if 'sha512-' + base64.b64encode(hashlib.sha512(data).digest()).decode() != lock['integrity']:
        raise RuntimeError('opencc-js 檔案雜湊不符，已停止')
    with tarfile.open(fileobj=io.BytesIO(data)) as archive_file:
        for member in archive_file.getmembers():
            if not member.isfile() or '..' in member.name.split('/'):
                continue
            path = target/member.name.split('/', 1)[1]
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(archive_file.extractfile(member).read())
    print('opencc-js', lock['resolved'].rsplit('-', 1)[-1].removesuffix('.tgz'), '已核對並安裝')
condition = {'vad10': 'whisper_live_vad_10s', 'full': 'whisper_no_prompt'}.get(CONDITIONS.split(',')[0], 'whisper_live_vad_10s')
result = subprocess.run([os.environ['NODE_BINARY'], '--max-old-space-size=8192', str(REPO/'scripts/mine-same-sound-terms.mjs'),
                         '--run', str(RUN), '--refs', str(DEV), '--condition', condition,
                         '--corpus', str(ARCHIVE/'life-study/text'), str(ARCHIVE/'recovery-version/text')],
                        capture_output=True, text=True, cwd=REPO)
if result.returncode:
    raise RuntimeError(result.stderr[-3000:])
(RUN/'term-candidates.md').write_text(result.stdout, encoding='utf-8')
print(result.stdout)
print('已存到：', RUN/'term-candidates.md')

## 請把兩樣東西貼回對話
1. 第 6 格最後的「**專名比對（不需逐字對齊）**」表格
2. 第 7 格的「**候選詞表**」

我會依表挑選要加入的專名（例如 分賜、神格、登寶座、賜生命的靈），重新產生讀音表後再用這組資料驗證「修好多、改錯 0」，最後才用 MEET 封存測試驗收。

欄位：**修好**＝辨識錯字被改成這個詞、且該篇文字確實有這個詞；**存疑**＝改了但該篇沒有這個詞；**被一般詞擋下**＝聽成的是一般詞（如「分次」）目前規則不改，需要你確認聚會中是否幾乎都是誤聽；**正確文字誤改**＝在恢復本／生命讀經正確文字裡會被改錯的次數，必須是 0。